# Ingestão de dados abertos ANEEL, ONS e IBGE

Este notebook executa a rotina de ingestão no Databricks e grava os arquivos no Volume Unity Catalog `/Volumes/mba/stage/dados_bruto/`. Os arquivos recebem o prefixo da plataforma e o processo usa intervalo entre requisições, pausas preventivas, retentativas e manifesto de rastreabilidade.

A implementação fica no arquivo `/Volumes/mba/stage/dados_bruto/ingest_dados_abertos.py`; o notebook apenas parametriza e chama `run_ingestion()`. O arquivo deve ser disponibilizado no Volume antes da execução.

In [0]:
from pathlib import Path
import importlib
import logging
import sys

VOLUME_ROOT = Path('/Volumes/mba/stage/dados_bruto')
MODULE_PATH = VOLUME_ROOT / 'ingest_dados_abertos.py'
if not MODULE_PATH.exists():
    raise FileNotFoundError(
        f'Arquivo obrigatório não encontrado: {MODULE_PATH}. '
        'Copie ingest_dados_abertos.py para a raiz do Volume antes de executar.'
    )

if str(VOLUME_ROOT) not in sys.path:
    sys.path.insert(0, str(VOLUME_ROOT))

import ingest_dados_abertos as _ingest_module
_ingest_module = importlib.reload(_ingest_module)
from ingest_dados_abertos import DEFAULT_OUTPUT_DIR, run_ingestion

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
)
logging.getLogger().setLevel(logging.INFO)
print(f'Módulo carregado de: {MODULE_PATH}')

## Parâmetros do notebook

Os parâmetros são widgets do Databricks e aceitam somente texto na interface. As conversões para número e booleano são feitas na célula seguinte. Para a primeira execução, recomenda-se manter `dry_run = false` somente depois de validar a descoberta com `dry_run = true`.

In [0]:
try:
    _dbutils = dbutils
except NameError:
    _dbutils = None

_defaults = {
    'delay_seconds': '2.0',
    'jitter_seconds': '1.0',
    'pause_every': '25',
    'pause_seconds': '20.0',
    'timeout_seconds': '120.0',
    'max_retries': '4',
    'max_files': '0',
    'force': 'false',
    'dry_run': 'false',
    'log_level': 'INFO',
}

if _dbutils is not None:
    def _ensure_text(name, default, label):
        try:
            _dbutils.widgets.get(name)
        except Exception:
            _dbutils.widgets.text(name, default, label)

    def _ensure_dropdown(name, default, choices, label):
        try:
            _dbutils.widgets.get(name)
        except Exception:
            _dbutils.widgets.dropdown(name, default, choices, label)

    _ensure_text('delay_seconds', _defaults['delay_seconds'], 'Intervalo mínimo (s)')
    _ensure_text('jitter_seconds', _defaults['jitter_seconds'], 'Jitter máximo (s)')
    _ensure_text('pause_every', _defaults['pause_every'], 'Pausa a cada N requisições')
    _ensure_text('pause_seconds', _defaults['pause_seconds'], 'Duração da pausa (s)')
    _ensure_text('timeout_seconds', _defaults['timeout_seconds'], 'Timeout (s)')
    _ensure_text('max_retries', _defaults['max_retries'], 'Retentativas')
    _ensure_text('max_files', _defaults['max_files'], 'Limite de arquivos; 0 = todos')
    _ensure_dropdown('force', _defaults['force'], ['false', 'true'], 'Forçar novo download')
    _ensure_dropdown('dry_run', _defaults['dry_run'], ['false', 'true'], 'Somente descobrir')
    _ensure_dropdown('log_level', _defaults['log_level'], ['INFO', 'WARNING', 'DEBUG'], 'Nível de log')
    print('Widgets Databricks prontos. Edite os valores no painel e execute novamente esta célula antes da execução.')
else:
    print('dbutils não disponível: usando os valores padrão. Este caso é útil apenas para teste local.')

In [0]:
def _get_widget(name):
    if _dbutils is None:
        return _defaults[name]
    return _dbutils.widgets.get(name)

def _to_bool(value):
    return str(value).strip().lower() in {'1', 'true', 't', 'yes', 'sim'}

config = {
    'output_dir': DEFAULT_OUTPUT_DIR,
    'delay_seconds': float(_get_widget('delay_seconds')),
    'jitter_seconds': float(_get_widget('jitter_seconds')),
    'pause_every': int(_get_widget('pause_every')),
    'pause_seconds': float(_get_widget('pause_seconds')),
    'timeout_seconds': float(_get_widget('timeout_seconds')),
    'max_retries': int(_get_widget('max_retries')),
    'max_files': int(_get_widget('max_files')),
    'force': _to_bool(_get_widget('force')),
    'dry_run': _to_bool(_get_widget('dry_run')),
    'log_level': _get_widget('log_level'),
}

logging.getLogger().setLevel(getattr(logging, config['log_level']))
print('Configuração selecionada:')
for _chave, _valor in config.items():
    print(f'  {_chave}: {_valor}')

## Executar a ingestão

A função retorna um dicionário com o resumo, os recursos descobertos, as falhas e o caminho do manifesto. O resultado fica disponível na variável `resultado` para inspeção posterior.

In [0]:
resultado = run_ingestion(**config)

print('Resumo da execução:')
for _chave, _valor in resultado['summary'].items():
    print(f'  {_chave}: {_valor}')
print('Manifestos por plataforma:')
for _plataforma, _manifesto in resultado['manifest_paths'].items():
    print(f'  {_plataforma}: {_manifesto}')

if resultado['failures']:
    print('Falhas encontradas:')
    for _falha in resultado['failures']:
        print(_falha)
    raise RuntimeError(
        f"A ingestão terminou com {len(resultado['failures'])} falha(s)."
    )

In [0]:
# No Databricks, transforma os resultados em DataFrames para inspeção no notebook.
try:
    display(spark.createDataFrame([resultado['summary']]))
    display(spark.createDataFrame(resultado['resources']))
except NameError:
    print(resultado['summary'])
    print(resultado['resources'][:10])